In [329]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from utils import *
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import wandb

### Read Data

In [ ]:
df = pd.read_csv('../../../data/creditcard.csv')

df = create_features(df)

scaler = StandardScaler()
df[['_log_amount']] = scaler.fit_transform(df[['_log_amount']])
df['hour_sin'] = np.sin(2 * np.pi * df['Hour_from_start_mod24']/24)
df['hour_cos'] = np.cos(2 * np.pi * df['Hour_from_start_mod24']/24)
df['time_diff'] = df['Time'].diff().fillna(0)
threshold = df['Amount'].quantile(0.95)  
df['is_high_amount'] = (df['Amount'] > threshold).astype(int)
df['is_rapid_transaction'] = (df['time_diff'] < 60).astype(int)

df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,Class,_log_amount,Hour_from_start_mod24,is_night_proxy,is_business_hours_proxy,hour_sin,hour_cos,time_diff,is_high_amount,is_rapid_transaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0,1.123062,0,1,0,0.0,1.0,0.0,0,1
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,0,-1.115298,0,1,0,0.0,1.0,0.0,0,1
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0,1.680981,0,1,0,0.0,1.0,1.0,1,1
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,0,1.008128,0,1,0,0.0,1.0,0.0,0,1
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0,0.669117,0,1,0,0.0,1.0,1.0,0,1


In [351]:
features = df.drop(['Time','Amount','Class','Hour_from_start_mod24'], axis=1).columns.tolist()
print(features)

['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', '_log_amount', 'is_night_proxy', 'is_business_hours_proxy', 'hour_sin', 'hour_cos', 'time_diff', 'is_high_amount', 'is_rapid_transaction']


In [352]:
X = df[features]
y = df['Class']

print(X.shape, y.shape)

(283726, 36) (283726,)


In [353]:
X_train, y_train, X_val, y_val, X_test, y_test = split_data(df, features, 'Class')

X_train: (181584, 36) y_train: (181584,)
X_val: (45396, 36) y_val: (45396,)
X_test: (56746, 36) y_test: (56746,)
Fraud rate in train: 0.001910961318177813
Fraud rate in test: 0.0013040566735981391


### Train Model 

In [367]:
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=7,
    min_samples_leaf=4,
    random_state=42,
    max_features='sqrt',
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf_model.fit(X_train,y_train)

p_test = rf_model.predict_proba(X_test)[:, 1]
p_val = rf_model.predict_proba(X_val)[:, 1]

rf_cal = CalibratedClassifierCV(
    rf_model,
    method='isotonic'
)

rf_cal.fit(X_val, y_val) 

p_test_cal = rf_cal.predict_proba(X_test)[:, 1]

result_rf = log_eval(y_test, p_test)
result_cal = log_eval(y_test, p_test_cal)

val_df = pd.DataFrame([
    {"model": "RF", **result_rf},
    {"model": "RF + Cal", **result_cal}
])

val_df


,model,threshold,Cost,ROC_AUC,PR_AUC,ece,brier
0,RF,0.048,2540.0,0.980542,0.816606,0.001071,0.000405
1,RF + Cal,0.036,3110.0,0.969879,0.797714,0.000144,0.000378


In [ ]:
# pos, neg = int((y_train==1).sum()), int((y_train==0).sum())
# spw = neg / max(1, pos)
# xgb_pipe = ImbPipeline(steps=[
#     ("model", XGBClassifier(
#         n_estimators=600, max_depth=4, learning_rate=0.05,
#         subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
#         tree_method="hist", random_state=SEED,
#         scale_pos_weight=spw, n_jobs=-1, eval_metric="aucpr"
#     ))
# ])
# xgb_pipe.fit(X_train, y_train)

# val_xgb_proba  = xgb_pipe.predict_proba(X_val)[:, 1]
# test_xgb_proba = xgb_pipe.predict_proba(X_test)[:, 1]

In [ ]:
# final_val_proba = 0.6 * val_xgb_proba + 0.4 * p_val

# best_th, _ = thr_min_cost(y_val, final_val_proba)

# final_test_proba = 0.6 * test_xgb_proba + 0.4 * p_test
# y_test_pred = (final_test_proba >= best_th).astype(int)

# rs_rf = log_eval(y_test, final_test_proba)
# rs = log_eval(y_val, final_val_proba)

# val_df = pd.DataFrame([
#     {"model": "RF + XBG test", **rs_rf},
#     {"model": "RF + XGB val", **rs}
# ])


# val_df

,model,threshold,Cost,ROC_AUC,PR_AUC,ece,brier
0,RF + XBG test,0.049,2745.0,0.985562,0.811767,0.001918,0.000427
1,RF + XGB val,0.142,2335.0,0.979499,0.762286,0.002017,0.000425


In [67]:
lr_model.fit(p_val.reshape(-1,1), y_val)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [68]:

p_test_cal = lr_model.predict_proba(p_test.reshape(-1, 1))[:, 1]

In [69]:
result_rf = log_eval(y_test, p_test)
result_cal = log_eval(y_test, p_test_cal)

val_df = pd.DataFrame([
    {"model": "RF", **result_rf},
    {"model": "RF + Cal", **result_cal}
])


val_df

,model,threshold,Cost,ROC_AUC,PR_AUC,ece,brier
0,RF,0.023,2925.0,0.950756,0.813337,0.000519,0.000419
1,RF + Cal,0.001,2930.0,0.950756,0.813337,0.000520,0.000575


In [ ]:
# start_run('Random forest và Calibrated')

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: miner337t (2351050051huy-tr-ng-i-h-c-m-th-nh-ph-h-ch-minh). Use `wandb login --relogin` to force relogin
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core
wandb: WARNING Unable to render HTML, can't import display from ipython.core


In [ ]:
# log_metrics(result_cal)